# PathoVision Model Training 🧬
This notebook downloads the **LC25000 Histopathology Dataset**, fine-tunes a **ResNet-50** CNN, and saves the trained `.pth` weights for your PathoVision project.

### Instructions:
1. Go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Run the cells one by one.
3. Paste your `KGAT_...` Kaggle API Token when prompted.
4. The trained `lung_colon_resnet50.pth` file will automatically download at the end!

In [ ]:
import os
import getpass

# It will prompt you for your token. Paste your KGAT_... token here and press Enter.
kaggle_token = getpass.getpass("Enter your KAGGLE_API_TOKEN (starts with KGAT_): ")
os.environ['KAGGLE_API_TOKEN'] = kaggle_token

print("Token set! Downloading dataset...")
!kaggle datasets download -d andrewmvd/lung-and-colon-cancer-histopathological-images --unzip
print("Download and unzip complete!")

In [ ]:
import shutil

# The dataset has two subfolders: lung_image_sets and colon_image_sets.
# We need to combine them into a single folder for PyTorch's ImageFolder.

data_dir = "lung_colon_image_set"
combined_dir = "combined_dataset"
os.makedirs(combined_dir, exist_ok=True)

# Move lung folders
lung_dir = os.path.join(data_dir, "lung_image_sets")
for class_folder in os.listdir(lung_dir):
    src = os.path.join(lung_dir, class_folder)
    dst = os.path.join(combined_dir, class_folder)
    if not os.path.exists(dst):
        shutil.move(src, combined_dir)

# Move colon folders
colon_dir = os.path.join(data_dir, "colon_image_sets")
for class_folder in os.listdir(colon_dir):
    src = os.path.join(colon_dir, class_folder)
    dst = os.path.join(combined_dir, class_folder)
    if not os.path.exists(dst):
        shutil.move(src, combined_dir)

print("✅ Classes organized:", os.listdir(combined_dir))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Transforms for Data Augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load full dataset
full_dataset = datasets.ImageFolder(combined_dir, transform=train_transform)
class_names = full_dataset.classes
print(f"Classes: {class_names}")

# Split into 80% Train, 20% Val
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# Load Pre-trained ResNet-50
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Replace final layer to output 5 classes instead of 1000
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
# We use AdamW optimizer for better generalization
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

In [ ]:
import time

# Training Loop
num_epochs = 5

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 10)

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
            dataloader = train_loader
        else:
            model.eval()
            dataloader = val_loader

        running_loss = 0.0
        running_corrects = 0
        start_time = time.time()

        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            
        if phase == 'train':
            scheduler.step()

        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_acc = running_corrects.double() / len(dataloader.dataset)
        time_elapsed = time.time() - start_time

        print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | Time: {time_elapsed:.0f}s')

print('\n✅ Training complete!')

In [ ]:
# Save the trained weights
torch.save(model.state_dict(), 'lung_colon_resnet50.pth')
print("Model saved successfully!")

from google.colab import files
files.download('lung_colon_resnet50.pth')
print("Downloading to your computer...")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Ensure model is in evaluation mode
model.eval()

y_true = []
y_pred = []

print("Running Evaluation on the 5,000 Validation Images... Please wait a minute.")

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.cpu().numpy()
        
        # Get predictions
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        preds = preds.cpu().numpy()
        
        y_true.extend(labels)
        y_pred.extend(preds)

# 1. Print Classification Report (Precision, Recall, F1-Score)
print('\n--- Classification Report ---')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# 2. Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Class', fontsize=12)
plt.ylabel('True Class', fontsize=12)
plt.title('PathoVision Confusion Matrix (Validation Set)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
